Google Drive Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check that Colab can see the files

In [2]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/Project2_SolarPV")

print("Folder exists:", folder.exists())

for filename in [
    "Plant_1_Generation_Data.csv",
    "Plant_1_Weather_Sensor_Data.csv",
    "Plant_2_Generation_Data.csv",
    "Plant_2_Weather_Sensor_Data.csv",
]:
    print(filename, "FOUND" if (folder / filename).exists() else "MISSING")

Folder exists: True
Plant_1_Generation_Data.csv FOUND
Plant_1_Weather_Sensor_Data.csv FOUND
Plant_2_Generation_Data.csv FOUND
Plant_2_Weather_Sensor_Data.csv FOUND


Import libraries and read the CSV files

In [3]:
import pandas as pd
import numpy as np

p1_gen_raw = pd.read_csv(folder / "Plant_1_Generation_Data.csv")
p1_wth_raw = pd.read_csv(folder / "Plant_1_Weather_Sensor_Data.csv")
p2_gen_raw = pd.read_csv(folder / "Plant_2_Generation_Data.csv")
p2_wth_raw = pd.read_csv(folder / "Plant_2_Weather_Sensor_Data.csv")

datasets = {
    "Plant 1 Generation": p1_gen_raw,
    "Plant 1 Weather": p1_wth_raw,
    "Plant 2 Generation": p2_gen_raw,
    "Plant 2 Weather": p2_wth_raw,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


Plant 1 Generation
Shape: (68778, 7)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.0,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.0,6987759.0



Plant 1 Weather
Shape: (3182, 6)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4135001,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,4135001,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,4135001,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0



Plant 2 Generation
Shape: (67698, 7)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,2020-05-15 00:00:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,9425.000000,2.429011e+06
1,2020-05-15 00:00:00,4136001,81aHJ1q11NBPMrL,0.0,0.0,0.000000,1.215279e+09
2,2020-05-15 00:00:00,4136001,9kRcWv60rDACzjR,0.0,0.0,3075.333333,2.247720e+09



Plant 2 Weather
Shape: (3259, 6)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4136001,iq8k7ZNt4Mwm3w0,27.004764,25.060789,0.0
1,2020-05-15 00:15:00,4136001,iq8k7ZNt4Mwm3w0,26.880811,24.421869,0.0
2,2020-05-15 00:30:00,4136001,iq8k7ZNt4Mwm3w0,26.682055,24.427290,0.0


Check and parse timestamps



In [4]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("First timestamps:", df["DATE_TIME"].head(3).tolist())
    print("Timestamp string lengths:")
    display(df["DATE_TIME"].astype(str).str.len().value_counts())


Plant 1 Generation
First timestamps: ['15-05-2020 00:00', '15-05-2020 00:00', '15-05-2020 00:00']
Timestamp string lengths:


,count
DATE_TIME,
16,68778



Plant 1 Weather
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:15:00', '2020-05-15 00:30:00']
Timestamp string lengths:


,count
DATE_TIME,
19,3182



Plant 2 Generation
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:00:00', '2020-05-15 00:00:00']
Timestamp string lengths:


,count
DATE_TIME,
19,67698



Plant 2 Weather
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:15:00', '2020-05-15 00:30:00']
Timestamp string lengths:


,count
DATE_TIME,
19,3259


Then parse each file with its known format

In [5]:
date_formats = {
    "Plant 1 Generation": "%d-%m-%Y %H:%M",
    "Plant 1 Weather": "%Y-%m-%d %H:%M:%S",
    "Plant 2 Generation": "%Y-%m-%d %H:%M:%S",
    "Plant 2 Weather": "%Y-%m-%d %H:%M:%S",
}

parsed = {}

for name, df in datasets.items():
    df = df.copy()
    df["DATE_TIME"] = pd.to_datetime(
        df["DATE_TIME"],
        format=date_formats[name],
        errors="raise",
    )
    parsed[name] = df

p1_gen = parsed["Plant 1 Generation"]
p1_wth = parsed["Plant 1 Weather"]
p2_gen = parsed["Plant 2 Generation"]
p2_wth = parsed["Plant 2 Weather"]

print("All timestamps converted successfully.")

All timestamps converted successfully.


Check each file’s rows, date range, missing values, and power

In [6]:
def summarize_dataset(name, df, value_column=None):
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print("Rows:", len(df))
    print("Date range:", df["DATE_TIME"].min(), "to", df["DATE_TIME"].max())

    if "SOURCE_KEY" in df.columns:
        print("Unique sources/inverters:", df["SOURCE_KEY"].nunique())

    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count"))

    if value_column:
        print(f"{value_column} summary:")
        display(df[value_column].describe().to_frame())
        print("Negative values:", (df[value_column] < 0).sum())

summarize_dataset("Plant 1 Generation", p1_gen, "AC_POWER")
summarize_dataset("Plant 1 Weather", p1_wth, "IRRADIATION")
summarize_dataset("Plant 2 Generation", p2_gen, "AC_POWER")
summarize_dataset("Plant 2 Weather", p2_wth, "IRRADIATION")


Plant 1 Generation
Rows: 68778
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 22
Missing values:


,missing_count
DATE_TIME,0
PLANT_ID,0
SOURCE_KEY,0
DC_POWER,0
AC_POWER,0
DAILY_YIELD,0
TOTAL_YIELD,0


AC_POWER summary:


,AC_POWER
count,68778.000000
mean,307.802752
std,394.396439
min,0.000000
25%,0.000000
50%,41.493750
75%,623.618750
max,1410.950000


Negative values: 0

Plant 1 Weather
Rows: 3182
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 1
Missing values:


...

Negative values: 0

Plant 2 Weather
Rows: 3259
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 1
Missing values:


,missing_count
DATE_TIME,0
PLANT_ID,0
SOURCE_KEY,0
AMBIENT_TEMPERATURE,0
MODULE_TEMPERATURE,0
IRRADIATION,0


IRRADIATION summary:


,IRRADIATION
count,3259.000000
mean,0.232737
std,0.312693
min,0.000000
25%,0.000000
50%,0.019040
75%,0.438717
max,1.098766


Negative values: 0


##Compare generation and weather timestamps
This checks how many timestamps are shared between generation and weather data.

---



In [7]:
def compare_timestamps(plant_name, generation_df, weather_df):
    generation_times = set(generation_df["DATE_TIME"].unique())
    weather_times = set(weather_df["DATE_TIME"].unique())

    common = generation_times & weather_times

    print(f"\n{plant_name}")
    print("Generation timestamps:", len(generation_times))
    print("Weather timestamps:", len(weather_times))
    print("Shared timestamps:", len(common))
    print("Generation without weather:", len(generation_times - weather_times))
    print("Weather without generation:", len(weather_times - generation_times))

compare_timestamps("Plant 1", p1_gen, p1_wth)
compare_timestamps("Plant 2", p2_gen, p2_wth)


Plant 1
Generation timestamps: 3158
Weather timestamps: 3182
Shared timestamps: 3157
Generation without weather: 1
Weather without generation: 25

Plant 2
Generation timestamps: 3259
Weather timestamps: 3259
Shared timestamps: 3259
Generation without weather: 0
Weather without generation: 0


##Compare inverter daytime records
This uses 06:00 inclusive through 18:00 exclusive.


In [8]:
def inverter_daytime_summary(plant_name, generation_df):
    daytime = generation_df[
        generation_df["DATE_TIME"].dt.hour.between(6, 17)
    ]

    summary = daytime.groupby("SOURCE_KEY")["AC_POWER"].agg(
        record_count="count",
        mean_power="mean",
        median_power="median",
        max_power="max",
        std_power="std",
    ).round(2)

    print(f"\n{plant_name}: daytime inverter power (06:00–17:59)")
    display(summary)

    print("Number of inverters:", len(summary))
    print(
        "Daytime records per inverter, minimum to maximum:",
        summary["record_count"].min(),
        "to",
        summary["record_count"].max(),
    )
    print(
        "Inverters with zero maximum daytime power:",
        (summary["max_power"] == 0).sum(),
    )

    return summary

p1_inverter_summary = inverter_daytime_summary("Plant 1", p1_gen)
p2_inverter_summary = inverter_daytime_summary("Plant 2", p2_gen)


Plant 1: daytime inverter power (06:00–17:59)


,record_count,mean_power,median_power,max_power,std_power
SOURCE_KEY,,,,,
1BY6WEcLGh8j5v7,1598,553.24,555.03,1300.17,341.46
1IF53ai7Xc0U56Y,1594,615.55,624.23,1394.51,363.91
3PZuoBAID5Wc2HD,1593,614.76,626.54,1399.22,364.74
7JYdWkrLSPkdwr4,1599,598.57,606.41,1384.87,356.48
McdE0feGgRqW7Ca,1598,612.13,624.53,1405.80,366.53
VHMLBKoKgIrUVDU,1599,613.27,623.74,1394.21,365.37
WRmjgnKYAwPKWDb,1593,602.27,612.42,1376.35,356.20
YxYtjZvoooNbGkE,1596,602.97,614.29,1410.53,359.01
ZnxXDlPa8U1GXgE,1598,610.01,623.63,1391.64,363.22


Number of inverters: 22
Daytime records per inverter, minimum to maximum: 1593 to 1599
Inverters with zero maximum daytime power: 0

Plant 2: daytime inverter power (06:00–17:59)


,record_count,mean_power,median_power,max_power,std_power
SOURCE_KEY,,,,,
4UPUqMRk7TRMgml,1595,541.34,497.58,1384.35,403.44
81aHJ1q11NBPMrL,1628,449.44,405.52,1347.66,384.91
9kRcWv60rDACzjR,1628,476.43,441.29,1361.67,385.88
Et9kgGMDl729KT4,1595,366.64,296.40,1172.65,336.56
IQ2d7wF4YD8zU1Q,1171,558.58,516.06,1383.46,384.32
LYwnQax7tkwH5Cb,1628,382.11,323.30,1265.67,345.32
LlT2YUhhzqhg5Sw,1628,478.14,442.63,1338.35,383.89
Mx2yZCDsyf6DPfv,1595,555.49,518.70,1377.15,402.98
NgDl19wMapZy17u,1171,535.10,494.31,1370.96,376.21


Number of inverters: 22
Daytime records per inverter, minimum to maximum: 1171 to 1628
Inverters with zero maximum daytime power: 0


##Compare plant-level daytime power
This adds together the power from all inverters at each timestamp. The raw sum can be low when inverters are missing. The scaled estimate uses the mean of the inverters reporting at each timestamp, multiplied by 22.

In [9]:
def plant_daytime_power(plant_name, generation_df):
    daytime = generation_df[
        generation_df["DATE_TIME"].dt.hour.between(6, 17)
    ]

    by_timestamp = daytime.groupby("DATE_TIME")["AC_POWER"].agg(
        reported_inverters="count",
        raw_sum="sum",
        mean_reported="mean",
    )

    by_timestamp["scaled_to_22"] = by_timestamp["mean_reported"] * 22

    print(f"\n{plant_name}: daytime plant AC_POWER")
    print("Timestamps:", len(by_timestamp))
    print("Timestamps with fewer than 22 inverters:",
          (by_timestamp["reported_inverters"] < 22).sum())
    display(by_timestamp[["raw_sum", "scaled_to_22"]].describe())

    return by_timestamp

p1_daytime_power = plant_daytime_power("Plant 1", p1_gen)
p2_daytime_power = plant_daytime_power("Plant 2", p2_gen)


Plant 1: daytime plant AC_POWER
Timestamps: 1600
Timestamps with fewer than 22 inverters: 10


,raw_sum,scaled_to_22
count,1600.000000,1600.000000
mean,13192.694370,13204.471509
std,7789.818926,7782.705449
min,0.000000,0.000000
25%,6153.666072,6184.893750
50%,13594.395536,13600.734822
75%,19230.952678,19230.952678
max,29150.212499,29150.212499



Plant 2: daytime plant AC_POWER
Timestamps: 1628
Timestamps with fewer than 22 inverters: 457


,raw_sum,scaled_to_22
count,1628.000000,1628.000000
mean,9979.719020,10661.233652
std,5523.661969,5884.771134
min,0.000000,0.000000
25%,5131.992381,5527.273141
50%,10804.296667,11514.346455
75%,14212.472262,15260.465000
max,25979.760476,25979.760476


##Aggregate Plant 1 and merge with weather
For consistency with the missing-inverter correction, this uses mean reported inverter power × 22 for the generation total.

In [10]:
p1_plant = (
    p1_gen.groupby("DATE_TIME")
    .agg(
        reported_inverters=("AC_POWER", "count"),
        mean_inverter_power=("AC_POWER", "mean"),
    )
    .reset_index()
)

p1_plant["AC_POWER"] = p1_plant["mean_inverter_power"] * 22
p1_plant = p1_plant.sort_values("DATE_TIME")

wth_clean = p1_wth[
    [
        "DATE_TIME",
        "AMBIENT_TEMPERATURE",
        "MODULE_TEMPERATURE",
        "IRRADIATION",
    ]
].copy()

if wth_clean["DATE_TIME"].duplicated().any():
    raise ValueError("Duplicate weather timestamps found.")

merged = pd.merge(
    p1_plant[["DATE_TIME", "AC_POWER", "reported_inverters"]],
    wth_clean,
    on="DATE_TIME",
    how="inner",
    validate="one_to_one",
).sort_values("DATE_TIME").reset_index(drop=True)

print("Generation timestamps before merge:", len(p1_plant))
print("Weather timestamps:", len(wth_clean))
print("Rows after merge:", len(merged))
print("Generation timestamps not matched:", len(p1_plant) - len(merged))
print("Columns:", merged.columns.tolist())
print("Missing values:")
display(merged.isna().sum().to_frame("missing_count"))
print("Date range:", merged["DATE_TIME"].min(), "to", merged["DATE_TIME"].max())
display(merged.head())

Generation timestamps before merge: 3158
Weather timestamps: 3182
Rows after merge: 3157
Generation timestamps not matched: 1
Columns: ['DATE_TIME', 'AC_POWER', 'reported_inverters', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
Missing values:


,missing_count
DATE_TIME,0
AC_POWER,0
reported_inverters,0
AMBIENT_TEMPERATURE,0
MODULE_TEMPERATURE,0
IRRADIATION,0


Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00


,DATE_TIME,AC_POWER,reported_inverters,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,0.0,21,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,0.0,21,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,0.0,21,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,0.0,21,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,0.0,22,24.621525,22.165423,0.0


Save the merged file to Google Drive

In [11]:
output_path = folder / "plant1_aggregated_merged.csv"
merged.to_csv(output_path, index=False)

print("Saved file to:", output_path)

Saved file to: /content/drive/MyDrive/Project2_SolarPV/plant1_aggregated_merged.csv
